Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking, "Why do parrots talk?" Let me start by recalling what I know about parrots and their ability to mimic human speech. First off, parrots are part of the Psittaciformes order, and they\'re known for their intelligence. I remember that they have a unique vocal organ called a syrinx, which allows them to produce a wide range of sounds. But why do they specifically mimic human speech?\n\nI think it\'s related to their social behavior. Parrots are social animals, and in the wild, they learn to communicate with each other through calls. Mimicking sounds could be a way to bond with their flock. When they\'re kept as pets, they might mimic humans to interact with their owners, trying to fit into their social group.\n\nAlso, their intelligence plays a role. Parrots are among the most intelligent birds, comparable to some primates. They can associate words with objects or actions. For example, a parrot might learn to say "hello" when someone

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I\'ll call the function with "Boston" as the location. Make sure the JSON is correctly formatted with the name and arguments. No other functions are available, so this should be straightforward.\n', 'tool_calls': [{'id': 'nny02sf77', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 154, 'total_tokens': 256, 'completion_time': 0.169497273, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.010655298, 'prompt_tokens_details': None, 'queue_time': 3.558835489, 'total_time': 0.180152571}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_cal

Tool Execution Loops


In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON and wrap the tool call in the required XML tags.\n', 'tool_calls': [{'id': 'v3cjy9t37', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 153, 'total_tokens': 251, 'completion_time': 0.163172367, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.02734729, 'prompt_tokens_details': None, 'queue_time': 2.047740628, 'total_time': 0.190519657}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2',